# VAE Architecture and Hyperparameter Sweep
**Baseline RNN: 0.217 | Best VAE: 0.253**

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Path setup
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Models and utilities from project structure
from models.vae import SideslipVAE, VAELoss
from utils.data_loadingV2 import load_data
from utils.band_mapping import load_band_mapping, calculate_band_mae
from utils.reproducibility_utils import seed_everything
from utils.visualization import plot_sweep_results, plot_method_comparison

SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")

Device: cuda
Project root: /home/mdesouky/workspace/mait_g2_drift_challenge


In [2]:
# Config
TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
SEQ_LEN = 20
BATCH_SIZE = 64
COLUMNS_TO_DROP = ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time']
VAL_RATIO = 0.15

In [3]:
# Load data
train_loader, val_loader, scaler, _, _ = load_data(
    train_path=str(TRAIN_PATH), val_ratio=VAL_RATIO, columns_to_drop=COLUMNS_TO_DROP,
    sequence_length=SEQ_LEN, target="sideSlip", threshold=True, threshold_value=3.0,
    threshold_column="vxCG", batchsize=BATCH_SIZE, scaling=True, seed=SEED, shuffle=True
)
band_dict = load_band_mapping()
print(f"Train: {len(train_loader)} batches, Val: {len(val_loader)} batches")

V5 Data Loader: Stratified Temporal Split

Processing 22 files...
Total groups after threshold: 42

Train: Many=90.3%, Medium=7.6%, Few=2.1%
Val:   Many=96.6%, Medium=2.3%, Few=1.1%

Train: 120,445 sequences | Val: 16,635 sequences

Train: 1882 batches, Val: 260 batches


In [4]:
def train_vae(config, train_loader, val_loader, band_dict, verbose=False):
    """Train VAE with two-stage training."""
    model = SideslipVAE(
        input_dim=config.get('input_dim', 22), hidden_dim=config.get('hidden_dim', 64),
        latent_dim=config.get('latent_dim', 16), seq_len=config.get('seq_len', 20),
        encoder_layers=config.get('encoder_layers', 1), decoder_layers=config.get('decoder_layers', 1)
    ).to(device)
    
    # Stage 1: L1 loss (30 epochs)
    loss_fn = VAELoss(alpha_recon=config.get('alpha_recon', 0.0), beta_kl=config.get('beta_kl', 0.01),
                      gamma_contrast=0.0, pred_threshold=0.0, pred_slope=0.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    
    for epoch in range(30):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss, _ = loss_fn(model(X), X, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
    
    # Stage 2: WeightedL1 (40 epochs)
    loss_fn = VAELoss(alpha_recon=config.get('alpha_recon', 0.0), beta_kl=config.get('beta_kl', 0.01),
                      gamma_contrast=config.get('gamma_contrast', 0.0), pred_threshold=2.0, pred_slope=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)
    
    best_mae, best_state = float('inf'), None
    for epoch in range(40):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss, _ = loss_fn(model(X), X, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        
        if (epoch + 1) % 10 == 0:
            metrics = evaluate(model, val_loader, band_dict)
            if verbose: print(f"  Epoch {epoch+1}: {metrics['overall_mae']:.4f}")
            if metrics['overall_mae'] < best_mae:
                best_mae = metrics['overall_mae']
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    if best_state: model.load_state_dict(best_state)
    return model, evaluate(model, val_loader, band_dict)

@torch.no_grad()
def evaluate(model, loader, band_dict):
    model.eval()
    preds, targets = [], []
    for X, y in loader:
        preds.extend(model(X.to(device))['y_pred'].cpu().numpy().flatten())
        targets.extend(y.numpy())
    return calculate_band_mae(np.array(preds), np.array(targets), band_dict)

## 1. Latent Dimension Sweep

In [5]:
latent_results = []
for latent_dim in [8, 16, 32, 64]:
    print(f"Training latent_dim={latent_dim}...")
    _, metrics = train_vae({'latent_dim': latent_dim, 'beta_kl': 0.01}, train_loader, val_loader, band_dict)
    print(f"  Overall={metrics['overall_mae']:.4f}, Few={metrics['few_mae']:.4f}")
    latent_results.append({'latent_dim': latent_dim, **metrics})

pd.DataFrame(latent_results)

Training latent_dim=8...
  Overall=0.1461, Few=0.2594
Training latent_dim=16...
  Overall=0.1513, Few=0.2388
Training latent_dim=32...
  Overall=0.1541, Few=0.1981
Training latent_dim=64...
  Overall=0.1531, Few=0.2115


,latent_dim,many_mae,many_count,many_pct,medium_mae,medium_count,medium_pct,few_mae,few_count,few_pct,overall_mae
0,8,0.130639,4230,25.428314,0.195234,95,0.571085,0.259427,18,0.108206,0.146068
1,16,0.135217,4230,25.428314,0.185662,95,0.571085,0.238815,18,0.108206,0.151280
2,32,0.141833,4230,25.428314,0.184392,95,0.571085,0.198085,18,0.108206,0.154127
3,64,0.136617,4230,25.428314,0.170629,95,0.571085,0.211475,18,0.108206,0.153051


## 2. Beta (KL Weight) Sweep

In [6]:
beta_results = []
for beta in [0.001, 0.005, 0.01, 0.02, 0.1]:
    print(f"Training beta={beta}...")
    _, metrics = train_vae({'latent_dim': 16, 'beta_kl': beta}, train_loader, val_loader, band_dict)
    print(f"  Overall={metrics['overall_mae']:.4f}, Few={metrics['few_mae']:.4f}")
    beta_results.append({'beta_kl': beta, **metrics})

pd.DataFrame(beta_results)

Training beta=0.001...
  Overall=0.1438, Few=0.2101
Training beta=0.005...
  Overall=0.1472, Few=0.2310
Training beta=0.01...
  Overall=0.1516, Few=0.1847
Training beta=0.02...
  Overall=0.1475, Few=0.2118
Training beta=0.1...
  Overall=0.1540, Few=0.2257


,beta_kl,many_mae,many_count,many_pct,medium_mae,medium_count,medium_pct,few_mae,few_count,few_pct,overall_mae
0,0.001,0.128160,4230,25.428314,0.189403,95,0.571085,0.210120,18,0.108206,0.143758
1,0.005,0.128403,4230,25.428314,0.208835,95,0.571085,0.230992,18,0.108206,0.147183
2,0.010,0.137228,4230,25.428314,0.168350,95,0.571085,0.184714,18,0.108206,0.151596
3,0.020,0.132145,4230,25.428314,0.198141,95,0.571085,0.211820,18,0.108206,0.147500
4,0.100,0.138668,4230,25.428314,0.179963,95,0.571085,0.225702,18,0.108206,0.154015


## 3. Architecture Sweep

In [7]:
arch_configs = [
    {'name': 'baseline', 'hidden_dim': 64, 'encoder_layers': 1},
    {'name': 'wide', 'hidden_dim': 128, 'encoder_layers': 1},
    {'name': 'deep', 'hidden_dim': 64, 'encoder_layers': 2},
    {'name': 'large', 'hidden_dim': 128, 'encoder_layers': 2},
]

arch_results = []
for cfg in arch_configs:
    name = cfg.pop('name')
    cfg.update({'latent_dim': 16, 'beta_kl': 0.005})
    print(f"Training {name}...")
    _, metrics = train_vae(cfg, train_loader, val_loader, band_dict)
    print(f"  Overall={metrics['overall_mae']:.4f}, Few={metrics['few_mae']:.4f}")
    arch_results.append({'name': name, **metrics})

pd.DataFrame(arch_results)

Training baseline...
  Overall=0.1497, Few=0.2353
Training wide...
  Overall=0.1481, Few=0.2270
Training deep...
  Overall=0.1451, Few=0.2126
Training large...
  Overall=0.1577, Few=0.2203


,name,many_mae,many_count,many_pct,medium_mae,medium_count,medium_pct,few_mae,few_count,few_pct,overall_mae
0,baseline,0.130571,4230,25.428314,0.187865,95,0.571085,0.235303,18,0.108206,0.149656
1,wide,0.130663,4230,25.428314,0.178794,95,0.571085,0.226953,18,0.108206,0.148103
2,deep,0.130633,4230,25.428314,0.164570,95,0.571085,0.212598,18,0.108206,0.145130
3,large,0.141487,4230,25.428314,0.185601,95,0.571085,0.220275,18,0.108206,0.157736


## Summary
**Optimal config:** `beta=0.005, alpha=0.0, gamma=0.0, latent_dim=16`

## 4. Predictions vs Ground Truth

In [8]:
# Train best config
best_config = {'latent_dim': 16, 'beta_kl': 0.005}
best_model, _ = train_vae(best_config, train_loader, val_loader, band_dict, verbose=True)

# Get predictions
best_model.eval()
val_preds, val_targets = [], []
with torch.no_grad():
    for X, y in val_loader:
        val_preds.extend(best_model(X.to(device))['y_pred'].cpu().numpy().flatten())
        val_targets.extend(y.numpy())

val_preds, val_targets = np.array(val_preds), np.array(val_targets)

# Using utils/visualization.py
from utils.visualization import plot_predictions_scatter
fig = plot_predictions_scatter(val_preds, val_targets, title='VAE: Predictions vs Ground Truth')
fig.show()

# Sweep results visualization
latent_df = pd.DataFrame(latent_results)
fig_latent = plot_sweep_results(latent_df, x_col='latent_dim', y_col='overall_mae',
                                 title='Latent Dimension Impact', highlight_optimal=True)
fig_latent.show()

  Epoch 10: 0.1533
  Epoch 20: 0.1644
  Epoch 30: 0.1551
  Epoch 40: 0.1557


## Interactive Predictions vs Ground Truth

In [9]:
# Interactive line plot showing predictions overlaid on ground truth
from utils.visualization import plot_predictions_vs_truth

fig_line = plot_predictions_vs_truth(
    val_preds, val_targets,
    title='VAE (Best Config): Predictions vs Ground Truth (Interactive)',
    max_samples=5000
)
fig_line.show()